In [1]:
from langchain_community.vectorstores import FAISS
from models.embedding import get_embedding
import os

os.environ['HF_DATASETS_OFFLINE'] = "1"
os.environ['TRANSFORMERS_OFFLINE'] = "1"
# 获取嵌入模型（必须与创建向量库时使用的一致）
embed_model = get_embedding(
    model="/data/fudawei/.cache/huggingface/hub/models--BAAI--bge-m3/snapshots/5617a9f61b028005a4858fdac845db406aefb181", 
    device="cuda:0"
)

# 加载 FAISS 向量库
vectorstore = FAISS.load_local(
    "/data/fudawei/graphRAG/data/vectorstore",
    embed_model.embed_model,  # 如果使用 EmbeddingWrapper，需要访问内部的 embed_model
    allow_dangerous_deserialization=True
)

In [2]:
# 使用向量库进行相似度搜索
results = vectorstore.similarity_search("你的查询文本", k=5)

for i, doc in enumerate(results, 1):
    print(f"{i}. [{doc.metadata.get('source')}] {doc.page_content[:100]}...")

1. [./data/book.txt]      distribution of Project Gutenberg™ works.
    

1.E.9. If you wish to charge a fee or distribut...
2. [./data/book.txt] . unless a copyright notice is included. Thus, we do not
necessarily keep eBooks in compliance with ...
3. [./data/book.txt]  paragraphs 1.E.1 through 1.E.7 and any
additional terms imposed by the copyright holder. Additional...
4. [./data/book.txt]         the use of Project Gutenberg™ works calculated using the method
        you already use to c...
5. [./data/book.txt]  EVEN IF YOU GIVE NOTICE OF THE POSSIBILITY OF SUCH
DAMAGE.

1.F.3. LIMITED RIGHT OF REPLACEMENT OR ...


In [3]:
for doc_id, doc in vectorstore.docstore._dict.items():
    print(f"Metadata: {doc.metadata}")
    print("-" * 40)

Metadata: {'source': './data/book.txt', 'chunk_id': 0}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 1}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 2}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 3}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 4}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 5}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 6}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 7}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 8}
----------------------------------------
Metadata: {'source': './data/book.txt', 'chunk_id': 9}
----------------------------------------
Metadata: {'source': './data/book.txt', 

In [4]:
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship


In [15]:
from langchain_core.documents import Document

document = Document(
    page_content="Hello, world!",
)

In [6]:
import requests
response = requests.post(
    f"http://localhost:8000/v1/chat/completions",
    json={
        "model": "Qwen/Qwen3.5-27B",
        "messages": [
            #{"role": "system", "content": args.system_prompt},
            {"role": "user", "content": "Who are you"}
        ],
        "chat_template_kwargs": {"enable_thinking": True},
    }
)
